# Tahap 3: Data Cleaning & Preprocessing
**Mata Kuliah:** Data Science 2026  
**Topik:** Pembersihan & Rekayasa Fitur King County Real Estate  
**Tujuan:** Mentransformasikan data mentah 147 kolom menjadi dataset analitik bersih 40 kolom yang bebas dari missing values, terbebas dari trailing whitespace, memiliki koordinat Latitude/Longitude, dan siap dilatih oleh model Machine Learning.

## 1. Import Pustaka

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Menambahkan direktori src ke sys.path untuk modul utilitas
sys.path.append(os.path.abspath(".."))
from src.utils import wa_north_to_latlon

## 2. Memuat Dataset Mentah

In [ ]:
DATA_PATH = "../data/raw/02_king_county_real_estate/cleansales.tab"
df = pd.read_csv(DATA_PATH, sep='\t', low_memory=False)
print(f"Data mentah awal: {df.shape[0]:,} baris x {df.shape[1]} kolom.")

## 3. Penanganan Data Duplikat & Pembersihan String Whitespace

In [ ]:
# Cek duplikat awal
print("Jumlah duplikasi baris awal:", df.duplicated().sum())

# Bersihkan trailing whitespace pada seluruh kolom teks/string
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

print("Pembersihan spasi berlebih pada kolom teks selesai.")

## 4. Penanganan Outlier Non-Pasar & Transaksi Anomali
Transaksi di bawah \$50.000 umumnya merupakan transfer kepemilikan keluarga, hibah, atau penyelesaian sengketa hukum (*non-arm's length transaction*), bukan harga pasar wajar.

In [ ]:
initial_count = len(df)
df = df[df['SalePrice'] >= 50000].copy()
filtered_out = initial_count - len(df)

print(f"Menyaring transaksi non-pasar (< $50,000): {filtered_out} baris dieliminasi.")
print(f"Sisa baris valid: {len(df):,} baris.")

## 5. Imputasi Spasial Missing ZipCode (K-Nearest Neighbors)
Memanfaatkan kelengkapan 100% koordinat `X` dan `Y` untuk merekonstruksi kode pos yang kosong secara geografis.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

missing_mask = df['ZipCode'].isnull() | (df['ZipCode'] == 'nan') | (df['ZipCode'] == '')
print(f"Jumlah ZipCode kosong yang perlu diimputasi: {missing_mask.sum():,}")

known_data = df[~missing_mask].copy()
known_data['ZipClean'] = known_data['ZipCode'].str[:5]

if missing_mask.sum() > 0:
    knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
    knn.fit(known_data[['X', 'Y']], known_data['ZipClean'])
    imputed_values = knn.predict(df.loc[missing_mask, ['X', 'Y']])
    df.loc[missing_mask, 'ZipCode'] = imputed_values

df['ZipCode'] = df['ZipCode'].astype(str).str[:5]
print("Imputasi ZipCode selesai. Sisa missing ZipCode:", df['ZipCode'].isnull().sum())

## 6. Transformasi Koordinat Spasial ke WGS84 (Latitude & Longitude)
Mengonversi proyeksi bidang datar Washington State Plane North (kaki) ke koordinat derajat bola bumi WGS84 untuk pemetaan interaktif.

In [ ]:
print("Mengonversi koordinat X, Y ke Latitude dan Longitude...")
lats, lons = [], []
for x_ft, y_ft in zip(df['X'], df['Y']):
    lat, lon = wa_north_to_latlon(x_ft, y_ft)
    lats.append(round(lat, 5))
    lons.append(round(lon, 5))

df['Latitude'] = lats
df['Longitude'] = lons
print(f"Koordinat terkonversi: Lat rentang [{df['Latitude'].min()}, {df['Latitude'].max()}], Lon rentang [{df['Longitude'].min()}, {df['Longitude'].max()}]")

## 7. Rekayasa Fitur (*Feature Engineering*) & Segmentasi Tier

In [ ]:
# 1. Usia properti saat transaksi
df['PropertyAge'] = np.maximum(0, df['salesYear'] - df['YrBuilt'])

# 2. Status renovasi
df['IsRenovated'] = (df['YrRenovated'] > 0).astype(int)

# 3. Total kamar mandi tertimbang
df['TotalBaths'] = df['BathFullCount'] + 0.75 * df['Bath3qtrCount'] + 0.5 * df['BathHalfCount']

# 4. Fitur pemandangan alam (Scenic View Indicator)
df['HasView'] = ((df['SeattleSkyline'] > 0) | 
                 (df['LakeWashington'] > 0) | 
                 (df['LakeSammamish'] > 0) | 
                 (df['PugetSound'] > 0) | 
                 (df['MtRainier'] > 0)).astype(int)

df['IsWaterfront'] = (df['WfntLocation'] > 0).astype(int)

# 5. Rasio harga per luas lantai
df['PricePerSqFt'] = df['SalePrice'] / np.maximum(1, df['SqFtTotLiving'])

# 6. Target Klasifikasi (PropertyTier)
df['PropertyTier'] = pd.cut(
    df['SalePrice'],
    bins=[-np.inf, 350000, 700000, np.inf],
    labels=['Entry-Level', 'Mid-Market', 'Luxury']
)

print("Distribusi Kelas Segmentasi Pasar (PropertyTier):")
print(df['PropertyTier'].value_counts())

## 8. Seleksi Fitur Kunci & Ekspor Data Bersih
Menyusutkan 147 kolom mentah menjadi 40 kolom bernilai prediktif tinggi dan menyimpannya ke `../data/processed/`.

In [ ]:
selected_cols = [
    'SalePrice', 'PropertyTier',
    'SqFtTotLiving', 'SqFtLot', 'Bedrooms', 'TotalBaths',
    'BathFullCount', 'BathHalfCount', 'Bath3qtrCount',
    'Stories', 'SqFtTotBasement', 'SqFtFinBasement', 'SqFtGarageAttached',
    'BldgGrade', 'Condition', 'YrBuilt', 'YrRenovated',
    'PropertyAge', 'IsRenovated',
    'ZipCode', 'X', 'Y', 'Latitude', 'Longitude', 'DistrictName',
    'IsWaterfront', 'HasView', 'SeattleSkyline', 'LakeWashington',
    'LakeSammamish', 'PugetSound', 'MtRainier',
    'TrafficNoise', 'AirportNoise',
    'LandVal', 'ImpsVal', 'PricePerSqFt', 'DocumentDate', 'salesYear'
]

clean_df = df[selected_cols].drop_duplicates().reset_index(drop=True)
output_path = "../data/processed/king_county_clean.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
clean_df.to_csv(output_path, index=False)

print(f"Data bersih berhasil disimpan ke {output_path}")
print(f"Dimensi akhir: {clean_df.shape[0]:,} baris x {clean_df.shape[1]} kolom.")
print("Total Missing Values di seluruh dataset bersih:", clean_df.isnull().sum().sum())

## 9. Ringkasan Perubahan Data
| Parameter Transformasi | Sebelum (Raw Data) | Sesudah (Processed Data) |
| :--- | :--- | :--- |
| **Jumlah Baris** | 22.847 baris | 22.748 baris (-99 baris non-pasar & duplikat) |
| **Jumlah Kolom** | 147 kolom | 40 kolom esensial |
| **Missing Values (ZipCode)** | 3.205 baris kosong | 0 (100% terimputasi via KNN spasial) |
| **Sistem Koordinat** | Hanya proyeksi kaki (*X, Y*) | Dilengkapi Latitude & Longitude derajat WGS84 |
| **Fitur Rekayasa Baru** | Tidak ada | `PropertyAge`, `IsRenovated`, `TotalBaths`, `HasView`, `PropertyTier` |
| **Integritas Target** | Skewness tinggi, rentang \$5k–\$9.4M | Bersih dari transaksi hibah nominal, siap transformasi log |